# SW-KAN Core Components

This notebook defines the reusable Stieltjes-Wigert polynomial basis and the PyTorch SW-KAN layer.

In [ ]:
import torch
import torch.nn as nn


def stieltjes_wigert_polynomials(x, degree, q_param):
    """Compute monic Stieltjes-Wigert polynomials P_0,...,P_degree.

    The recurrence is:
        x P_n(x) = P_{n+1}(x) + b_n P_n(x) + a_n^2 P_{n-1}(x)
    where
        b_n = q^(-2n - 3/2) * (1 + q - q^(n+1))
        a_n^2 = q^(-4n) * (1 - q^n)

    q is constrained to (0.005, 0.995) for numerical stability. Inputs are
    mapped to the positive support of the Stieltjes-Wigert weight.
    """
    q = torch.sigmoid(q_param) * 0.99 + 0.005
    x_pos = torch.exp(2.0 * torch.tanh(x))

    pm1 = torch.zeros_like(x_pos)
    p0 = torch.ones_like(x_pos)
    polynomials = [p0]

    if degree >= 1:
        p1 = x_pos - q ** (-1.5)
        polynomials.append(p1)
        pm1, p0 = p0, p1

    for n in range(1, degree):
        bn = q ** (-2.0 * n - 1.5) * (1.0 + q - q ** (n + 1))
        an2 = q ** (-4.0 * n) * (1.0 - q ** n)
        p1 = x_pos * p0 - bn * p0 - an2 * pm1
        polynomials.append(p1)
        pm1, p0 = p0, p1

    return torch.stack(polynomials, dim=-1)


class StieltjesWigertKANLayer(nn.Module):
    """KAN layer based on Stieltjes-Wigert monic polynomial features."""

    def __init__(self, input_dim, output_dim, degree=3, learnable_q=True):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.degree = degree
        self.weights = nn.Parameter(
            torch.randn(output_dim, input_dim, degree + 1) * 0.02
        )
        if learnable_q:
            self.q_param = nn.Parameter(torch.tensor(0.0))
        else:
            self.register_buffer('q_param', torch.tensor(0.0))

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.shape[0], -1)
        polynomials = stieltjes_wigert_polynomials(x, self.degree, self.q_param)
        return torch.einsum('bid,oid->bo', polynomials, self.weights)


def count_parameters(model, verbose=True):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if verbose:
        print(f"Total trainable parameters: {total:,}")
        for name, parameter in model.named_parameters():
            if parameter.requires_grad:
                print(f"  {name:40s} {parameter.numel():>10,}  {list(parameter.shape)}")
    return total
